In [1]:
import sqlite3
import pandas as pd

# Создаём базу данных в памяти
conn = sqlite3.connect(':memory:')

# Загружаем наш датасет в БД
url = "https://raw.githubusercontent.com/plotly/datasets/master/gapminder_with_codes.csv"
df = pd.read_csv(url)
df.to_sql('gapminder', conn, index=False, if_exists='replace')

print("База данных готова ✅")
print(f"Таблица 'gapminder' создана с {len(df)} строками")


База данных готова ✅
Таблица 'gapminder' создана с 1704 строками


In [2]:
# Простая функция для удобного выполнения запросов
def sql(query):
    return pd.read_sql(query, conn)

# Первый запрос
sql("SELECT country, year, gdpPercap, lifeExp FROM gapminder WHERE year = 2007 ORDER BY gdpPercap DESC LIMIT 10")
sql("""
select * from gapminder
where country = 'Norway'
""")

,country,continent,year,lifeExp,pop,gdpPercap,iso_alpha,iso_num
0,Norway,Europe,1952,72.670,3327728,10095.42172,NOR,578
1,Norway,Europe,1957,73.440,3491938,11653.97304,NOR,578
2,Norway,Europe,1962,73.470,3638919,13450.40151,NOR,578
3,Norway,Europe,1967,74.080,3786019,16361.87647,NOR,578
4,Norway,Europe,1972,74.340,3933004,18965.05551,NOR,578
5,Norway,Europe,1977,75.370,4043205,23311.34939,NOR,578
6,Norway,Europe,1982,75.970,4114787,26298.63531,NOR,578
7,Norway,Europe,1987,75.890,4186147,31540.97480,NOR,578
8,Norway,Europe,1992,77.320,4286357,33965.66115,NOR,578
9,Norway,Europe,1997,78.320,4405672,41283.16433,NOR,578


In [3]:
sql("""
    SELECT continent, 
           COUNT(DISTINCT country) as кол_стран,
           ROUND(AVG(gdpPercap), 0) as средний_ВВП,
           ROUND(AVG(lifeExp), 1) as средняя_жизнь
    FROM gapminder 
    WHERE year = 2007
    GROUP BY continent
    ORDER BY средний_ВВП DESC
""")

,continent,кол_стран,средний_ВВП,средняя_жизнь
0,Oceania,2,29810.0,80.7
1,Europe,30,25054.0,77.6
2,Asia,33,12473.0,70.7
3,Americas,25,11003.0,73.6
4,Africa,52,3089.0,54.8


In [4]:
sql("""
    SELECT 
        country,
        continent,
        gdpPercap,
        ROUND(AVG(gdpPercap) OVER (PARTITION BY continent), 0) as среднее_континента,
        ROUND(gdpPercap - AVG(gdpPercap) OVER (PARTITION BY continent), 0) as отклонение
    FROM gapminder
    WHERE year = 2007
    ORDER BY continent, gdpPercap DESC
""")

,country,continent,gdpPercap,среднее_континента,отклонение
0,Gabon,Africa,13206.484520,3089.0,10117.0
1,Botswana,Africa,12569.851770,3089.0,9481.0
2,Equatorial Guinea,Africa,12154.089750,3089.0,9065.0
3,Libya,Africa,12057.499280,3089.0,8968.0
4,Mauritius,Africa,10956.991120,3089.0,7868.0
...,...,...,...,...,...
137,Turkey,Europe,8458.276384,25054.0,-16596.0
138,Bosnia and Herzegovina,Europe,7446.298803,25054.0,-17608.0
139,Albania,Europe,5937.029526,25054.0,-19117.0
140,Australia,Oceania,34435.367440,29810.0,4625.0


In [10]:
sql("""
select 
country,
continent,
lifeExp,
max(lifeExp) over (partition by continent) as maxlifeExp,
lifeExp -  max(lifeExp) over (partition by continent) as difference
from gapminder
where year = 2007
order by continent desc
""")

,country,continent,lifeExp,maxlifeExp,difference
0,Australia,Oceania,81.235,81.235,0.000
1,New Zealand,Oceania,80.204,81.235,-1.031
2,Albania,Europe,76.423,81.757,-5.334
3,Austria,Europe,79.829,81.757,-1.928
4,Belgium,Europe,79.441,81.757,-2.316
...,...,...,...,...,...
137,Togo,Africa,58.420,76.442,-18.022
138,Tunisia,Africa,73.923,76.442,-2.519
139,Uganda,Africa,51.542,76.442,-24.900
140,Zambia,Africa,42.384,76.442,-34.058


In [22]:
sql("""
select * from 
(select
country,
continent,
lifeExp,
rank() over (partition by continent order by lifeExp desc) as rank
from gapminder
where year = 2007) as ranked
where rank <=3
order by continent, rank

""")

,country,continent,lifeExp,rank
0,Reunion,Africa,76.442,1
1,Libya,Africa,73.952,2
2,Tunisia,Africa,73.923,3
3,Canada,Americas,80.653,1
4,Costa Rica,Americas,78.782,2
5,Puerto Rico,Americas,78.746,3
6,Japan,Asia,82.603,1
7,"Hong Kong, China",Asia,82.208,2
8,Israel,Asia,80.745,3
9,Iceland,Europe,81.757,1


In [48]:
sql("""
select
country,
continent,
lifeExp,
gdpPercap,
lag(gdpPercap, 5) over (partition by country order by year) as gdpPercap_lastyear
from gapminder
where year = 2007

""")

,country,continent,lifeExp,gdpPercap,gdpPercap_lastyear
0,Afghanistan,Asia,43.828,974.580338,None
1,Albania,Europe,76.423,5937.029526,None
2,Algeria,Africa,72.301,6223.367465,None
3,Angola,Africa,42.731,4797.231267,None
4,Argentina,Americas,75.320,12779.379640,None
...,...,...,...,...,...
137,Vietnam,Asia,74.249,2441.576404,None
138,West Bank and Gaza,Asia,73.422,3025.349798,None
139,"Yemen, Rep.",Asia,62.698,2280.769906,None
140,Zambia,Africa,42.384,1271.211593,None


In [4]:
sql("""
    WITH ranked AS (
        SELECT 
            country,
            continent,
            lifeExp,
            gdpPercap,
            RANK() OVER (PARTITION BY continent ORDER BY lifeExp DESC) as ранг
        FROM gapminder 
        WHERE year = 2007
    ),
    averages AS (
        SELECT 
            continent,
            ROUND(AVG(lifeExp), 1) as avg_lifeExp,
            ROUND(AVG(gdpPercap), 0) as avg_gdp
        FROM gapminder
        WHERE year = 2007
        GROUP BY continent
    )
    SELECT 
        r.country,
        r.continent,
        r.lifeExp,
        r.ранг,
        a.avg_lifeExp as среднее_континента
    FROM ranked r
    JOIN averages a ON r.continent = a.continent
    WHERE r.ранг <= 3
    ORDER BY r.continent, r.ранг
""")

,country,continent,lifeExp,ранг,среднее_континента
0,Reunion,Africa,76.442,1,54.8
1,Libya,Africa,73.952,2,54.8
2,Tunisia,Africa,73.923,3,54.8
3,Canada,Americas,80.653,1,73.6
4,Costa Rica,Americas,78.782,2,73.6
5,Puerto Rico,Americas,78.746,3,73.6
6,Japan,Asia,82.603,1,70.7
7,"Hong Kong, China",Asia,82.208,2,70.7
8,Israel,Asia,80.745,3,70.7
9,Iceland,Europe,81.757,1,77.6


In [3]:
sql("""
    WITH lagging AS (
        SELECT 
            country, 
            year,
            gdpPercap,
            LAG(gdpPercap, 1) OVER (PARTITION BY country ORDER BY year) as lastgdp,
            gdpPercap - LAG(gdpPercap, 1) OVER (PARTITION BY country ORDER BY year) as diff
        FROM gapminder
        
    )
    SELECT * FROM lagging
    where year between 1952 and 1957
    ORDER BY diff desc
    limit 3
""")

,country,year,gdpPercap,lastgdp,diff
0,Kuwait,1957,113523.13290,108382.352900,5140.780000
1,Switzerland,1957,17909.48973,14734.232750,3175.256980
2,Germany,1957,10187.82665,7144.114393,3043.712257
